<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/CheckStrategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MAE50 & MAE 200

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time

# ==========================================
# НАСТРОЙКИ БЭКТЕСТА
# ==========================================
TICKER = "SBER"        # Тикер (например, SBER, GAZP, LKOH). Для индекса ставьте "IMOEX"
DAYS_BACK = 3 * 365    # Глубина истории (3 года)
INTERVAL = 60          # 60 = часовой таймфрейм
RR_RATIOS = [0.5, 1, 1.5, 2, 3, 5]         # Risk/Reward

# ==========================================
# 1. Загрузка исторический данных (Свечи)
# ==========================================
def fetch_history(ticker, days_back, interval):
    print(f"📥 Загружаем историю для {ticker} за {days_back} дней (Часовики)...")
    till = datetime.now()
    since = till - timedelta(days=days_back)

    # MOEX API по-разному отдает акции и индексы. Делаем проверку:
    if ticker == "IMOEX":
        url = f"https://iss.moex.com/iss/engines/stock/markets/index/boards/SNDX/securities/{ticker}/candles.json"
    else:
        url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"

    all_rows, start, columns = [], 0, None

    with tqdm(desc="Скачивание страниц") as pbar:
        while True:
            params = {
                "from": since.strftime("%Y-%m-%d"),
                "till": till.strftime("%Y-%m-%d"),
                "interval": interval,
                "start": start,
                "iss.meta": "off"
            }
            r = requests.get(url, params=params, timeout=15)
            if r.status_code != 200:
                break
            js = r.json().get("candles", {})
            rows = js.get("data", [])
            if columns is None:
                columns = js.get("columns", [])

            if not rows:
                break

            all_rows.extend(rows)
            pbar.update(1)

            if len(rows) < 500:
                break
            start += len(rows)
            time.sleep(0.1)

    if not all_rows:
        print("❌ Нет данных!")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "begin": "Date"})
    df = df[["Date", "Open", "High", "Low", "Close"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

    return df

# ==========================================
# 2. Добавление индикаторов (EMA50, EMA200)
# ==========================================
def add_indicators(df):
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    return df

# ==========================================
# 3. ЛОГИКА СТРАТЕГИИ И БЭКТЕСТ
# ==========================================
def run_backtest(df):
    print("⚙️ Запускаем симуляцию торгов...")

    trades = []

    # Состояния логики
    in_trade = False
    in_pullback = False
    pullback_low = float('inf')

    # Параметры текущей сделки
    entry_price = 0
    sl = 0
    tp = 0
    entry_date = None

    for i in range(200, len(df)):
        current = df.iloc[i]

        # -----------------------------------
        # ЕСЛИ МЫ В СДЕЛКЕ -> Проверяем выход
        # -----------------------------------
        if in_trade:
            # Консервативный подход: если в одной свече пробило и стоп и тейк, считаем что это стоп-лосс (чтобы не завышать результаты)
            if current["Low"] <= sl:
                trades.append({
                    "Вход (Дата)": entry_date,
                    "Выход (Дата)": current["Date"],
                    "Цена входа": round(entry_price, 2),
                    "Стоп-Лосс": round(sl, 2),
                    "Тейк-Профит": round(tp, 2),
                    "Результат": "❌ LOSS"
                })
                in_trade = False

            elif current["High"] >= tp:
                trades.append({
                    "Вход (Дата)": entry_date,
                    "Выход (Дата)": current["Date"],
                    "Цена входа": round(entry_price, 2),
                    "Стоп-Лосс": round(sl, 2),
                    "Тейк-Профит": round(tp, 2),
                    "Результат": "✅ WIN"
                })
                in_trade = False

            continue # Если мы в сделке, новые сигналы не ищем

        # -----------------------------------
        # ЕСЛИ МЫ БЕЗ ПОЗИЦИИ -> Ищем сигнал
        # -----------------------------------

        # Условие 1: Глобальный тренд (50 > 200)
        is_uptrend = current["EMA50"] > current["EMA200"]

        if not is_uptrend:
            in_pullback = False # Если тренд сменился, отменяем ожидание
            continue

        # Условие 2: Мы НЕ в откате, ждем пробития EMA50 вниз красной свечой
        if not in_pullback:
            is_red = current["Close"] < current["Open"]
            if is_red and current["Close"] < current["EMA50"] and current["Close"] > current["EMA200"]:
                in_pullback = True
                pullback_low = current["Low"] # Запоминаем минимум

        # Условие 3: Мы уже в откате (цена между 50 и 200)
        else:
            # Обновляем абсолютный минимум отката (для стоп-лосса)
            pullback_low = min(pullback_low, current["Low"])

            # Если цена провалилась ниже 200 EMA - откат слишком глубокий, отменяем сетап
            if current["Close"] < current["EMA200"]:
                in_pullback = False
                continue

            # ИЩЕМ СИГНАЛ: Зеленая свеча закрывается ВЫШЕ EMA50
            is_green = current["Close"] > current["Open"]
            if is_green and current["Close"] > current["EMA50"]:
                # СИГНАЛ НА ПОКУПКУ!
                in_trade = True
                in_pullback = False
                entry_price = current["Close"]
                entry_date = current["Date"]

                # Расчет Риск-менеджмента
                sl = pullback_low
                risk = entry_price - sl

                # Защита от нулевого риска (если свеча плоская)
                if risk <= 0:
                    risk = entry_price * 0.005
                    sl = entry_price - risk

                tp = entry_price + (risk * RR_RATIO)

    return pd.DataFrame(trades)

# ==========================================
# 4. ЗАПУСК
# ==========================================
df = fetch_history(TICKER, DAYS_BACK, INTERVAL)

if not df.empty:
    df = add_indicators(df)


    for i in RR_RATIOS:
      RR_RATIO = i

      trades_df = run_backtest(df)

      print("\n" + "="*50)
      print(f"📊 ИТОГИ БЭКТЕСТА: {TICKER} (Часовик, 3 года)")
      print("="*50)

      if trades_df.empty:
          print("Сделок по данной стратегии не найдено.")
      else:
          total_trades = len(trades_df)
          wins = len(trades_df[trades_df["Результат"] == "✅ WIN"])
          losses = len(trades_df[trades_df["Результат"] == "❌ LOSS"])
          winrate = (wins / total_trades) * 100

          print(f"Всего сделок: {total_trades}")
          print(f"Прибыльных (WIN): {wins}")
          print(f"Убыточных (LOSS): {losses}")
          print(f"Винрейт: {winrate:.1f}%")
          print(f"Соотношение Риск/Прибыль (R:R): 1:{RR_RATIO}")

          # Оценка профита в "R" (Единицах риска)
          # Каждая победа приносит 2R, каждый убыток забирает 1R
          total_r = (wins * RR_RATIO) - losses
          print(f"Чистый профит в единицах риска: {total_r:.1f} R")
          print("\n📝 ПОСЛЕДНИЕ 15 СДЕЛОК:")
          print(trades_df.tail(15).to_string(index=False))

📥 Загружаем историю для SBER за 1095 дней (Часовики)...


Скачивание страниц: 28it [00:23,  1.17it/s]


⚙️ Запускаем симуляцию торгов...

📊 ИТОГИ БЭКТЕСТА: SBER (Часовик, 3 года)
Всего сделок: 122
Прибыльных (WIN): 74
Убыточных (LOSS): 48
Винрейт: 60.7%
Соотношение Риск/Прибыль (R:R): 1:0.5
Чистый профит в единицах риска: -11.0 R

📝 ПОСЛЕДНИЕ 15 СДЕЛОК:
        Вход (Дата)        Выход (Дата)  Цена входа  Стоп-Лосс  Тейк-Профит Результат
2026-03-12 08:00:00 2026-03-12 17:00:00      315.72     314.43       316.36     ✅ WIN
2026-03-17 07:00:00 2026-03-17 13:00:00      316.86     315.72       317.43     ✅ WIN
2026-03-19 16:00:00 2026-03-19 22:00:00      320.04     319.21       320.46     ✅ WIN
2026-03-20 20:00:00 2026-03-23 07:00:00      320.35     320.01       320.52    ❌ LOSS
2026-04-09 10:00:00 2026-04-09 11:00:00      318.77     317.93       319.19     ✅ WIN
2026-04-10 08:00:00 2026-04-10 10:00:00      318.55     317.51       319.07    ❌ LOSS
2026-04-13 08:00:00 2026-04-13 10:00:00      317.86     317.31       318.14    ❌ LOSS
2026-04-20 13:00:00 2026-04-20 14:00:00      323.80     323.

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. НАСТРОЙКИ БЭКТЕСТА
# ==========================================
TICKER = "SBER"
DAYS_BACK = 3 * 365
INTERVAL = 60
RR_RATIOS = [0.5, 1, 1.5, 2.0, 2.5, 3.0]         # Далекий тейк-профит (1:3), чтобы дать тренду развиться

# ==========================================
# 2. УМНОЕ УПРАВЛЕНИЕ ПОЗИЦИЕЙ (SMART MANAGEMENT)
# ==========================================
# Перевод в безубыток
USE_BREAKEVEN = True
BREAKEVEN_TRIGGER_Rs = [0.5, 1, 1.5, 2.0, 2.5, 3.0]  # При достижении профита в 1R переводим стоп в ноль (на цену входа)

# Трейлинг-стоп
USE_TRAILING_STOP = True
TRAILING_DISTANCE_Rs = [0.5, 1, 1.5, 2.0, 2.5, 3.0]  # Держим стоп на расстоянии 1R от локального максимума цены

# ==========================================
# ЗАГРУЗКА И ИНДИКАТОРЫ (Скрыл лишние принты)
# ==========================================
def fetch_history(ticker, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
    all_rows, start, columns = [], 0, None
    with tqdm(desc="Скачивание истории", leave=False) as pbar:
        while True:
            params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": interval, "start": start, "iss.meta": "off"}
            r = requests.get(url, params=params, timeout=15)
            if r.status_code != 200: break
            js = r.json().get("candles", {})
            rows = js.get("data", [])
            if columns is None: columns = js.get("columns", [])
            if not rows: break
            all_rows.extend(rows)
            pbar.update(1)
            if len(rows) < 500: break
            start += len(rows)
            time.sleep(0.05)
    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "begin": "Date"})
    df = df[["Date", "Open", "High", "Low", "Close"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    return df

# ==========================================
# 3. ЛОГИКА СТРАТЕГИИ
# ==========================================
def run_backtest(df):
    trades = []
    in_trade, in_pullback = False, False
    pullback_low = float('inf')

    entry_price, sl, tp, initial_risk = 0, 0, 0, 0
    entry_date = None
    highest_price = 0  # Для трейлинга

    for i in range(200, len(df)):
        current = df.iloc[i]

        # --- УПРАВЛЕНИЕ ОТКРЫТОЙ СДЕЛКОЙ ---
        if in_trade:
            # Обновляем локальный максимум цены с момента входа
            highest_price = max(highest_price, current["High"])

            # Считаем текущий плавающий профит в единицах риска (R)
            current_profit_R = (highest_price - entry_price) / initial_risk

            # ЛОГИКА 1: Перевод в Безубыток
            if USE_BREAKEVEN and current_profit_R >= BREAKEVEN_TRIGGER_R:
                if sl < entry_price: # Если стоп еще не в безубытке
                    sl = entry_price # Двигаем стоп на цену входа

            # ЛОГИКА 2: Трейлинг-стоп (подтягиваем стоп за ценой)
            if USE_TRAILING_STOP and current_profit_R > TRAILING_DISTANCE_R:
                new_sl = highest_price - (initial_risk * TRAILING_DISTANCE_R)
                if new_sl > sl: # Стоп двигаем только ВВЕРХ
                    sl = new_sl

            # Проверка выхода по Стопу (или Трейлингу/Безубытку)
            if current["Low"] <= sl:
                # Определяем результат
                if sl == entry_price:
                    res_text = "⚪ BREAKEVEN"
                    pnl_r = 0
                elif sl > entry_price:
                    res_text = "🟢 TRAIL WIN"
                    pnl_r = (sl - entry_price) / initial_risk
                else:
                    res_text = "🔴 LOSS"
                    pnl_r = -1.0

                trades.append({
                    "Вход (Дата)": entry_date, "Выход (Дата)": current["Date"],
                    "Вход": round(entry_price, 2), "Выход": round(sl, 2),
                    "Результат": res_text, "Профит (R)": round(pnl_r, 2)
                })
                in_trade = False

            # Проверка выхода по жесткому Тейку (если он вообще достижим)
            elif current["High"] >= tp:
                trades.append({
                    "Вход (Дата)": entry_date, "Выход (Дата)": current["Date"],
                    "Вход": round(entry_price, 2), "Выход": round(tp, 2),
                    "Результат": "🏆 FULL WIN", "Профит (R)": RR_RATIO
                })
                in_trade = False

            continue

        # --- ПОИСК СИГНАЛА НА ВХОД ---
        is_uptrend = current["EMA50"] > current["EMA200"]
        if not is_uptrend:
            in_pullback = False
            continue

        if not in_pullback:
            if current["Close"] < current["Open"] and current["Close"] < current["EMA50"] and current["Close"] > current["EMA200"]:
                in_pullback = True
                pullback_low = current["Low"]
        else:
            pullback_low = min(pullback_low, current["Low"])
            if current["Close"] < current["EMA200"]:
                in_pullback = False
                continue

            if current["Close"] > current["Open"] and current["Close"] > current["EMA50"]:
                in_trade = True
                in_pullback = False
                entry_price = current["Close"]
                entry_date = current["Date"]

                initial_risk = entry_price - pullback_low
                if initial_risk <= 0: initial_risk = entry_price * 0.005

                sl = entry_price - initial_risk
                tp = entry_price + (initial_risk * RR_RATIO)
                highest_price = entry_price

    return pd.DataFrame(trades)

# ==========================================
# 4. ЗАПУСК И ОТЧЕТ
# ==========================================
print("🔄 Начинаем тестирование 'Умного управления'...")
df = fetch_history(TICKER, DAYS_BACK, INTERVAL)

if not df.empty:
    for i in RR_RATIOS:
      RR_RATIO = i
      for j in BREAKEVEN_TRIGGER_Rs:
        BREAKEVEN_TRIGGER_R = j
        for k in TRAILING_DISTANCE_Rs:
          TRAILING_DISTANCE_R = k

          trades_df = run_backtest(df)

          print("\n" + "="*60)
          print(f"📊 ИТОГИ БЭКТЕСТА (SMART MANAGEMENT): {TICKER}")
          print(f"Настройки: Б/У при +{BREAKEVEN_TRIGGER_R}R | Трейлинг: {USE_TRAILING_STOP} | Тейк: {RR_RATIO}R")
          print("="*60)

          if trades_df.empty:
              print("Сделок не найдено.")
          else:
              total = len(trades_df)
              full_wins = len(trades_df[trades_df["Результат"] == "🏆 FULL WIN"])
              trail_wins = len(trades_df[trades_df["Результат"] == "🟢 TRAIL WIN"])
              breakevens = len(trades_df[trades_df["Результат"] == "⚪ BREAKEVEN"])
              losses = len(trades_df[trades_df["Результат"] == "🔴 LOSS"])

              # Общий профит в R (сумма колонки)
              total_pnl_r = trades_df["Профит (R)"].sum()

              print(f"Всего сделок: {total}")
              print(f"Закрыто по полному Тейку: {full_wins}")
              print(f"Закрыто по Трейлинг-стопу (в плюс): {trail_wins}")
              print(f"Закрыто по Безубытку (в ноль): {breakevens}")
              print(f"Полные убытки: {losses}")
              print("-"*60)
              print(f"💰 ЧИСТЫЙ ПРОФИТ: {total_pnl_r:.2f} R")
              print("="*60)
              print("\n📝 ПОСЛЕДНИЕ 15 СДЕЛОК:")
              print(trades_df.tail(15).to_string(index=False))

🔄 Начинаем тестирование 'Умного управления'...


Выходные данные были обрезаны до нескольких последних строк (5000).
2026-04-13 08:00:00 2026-04-13 10:00:00 317.86 317.31      🔴 LOSS       -1.00
2026-04-20 13:00:00 2026-04-20 14:00:00 323.80 324.10 🟢 TRAIL WIN        0.46
2026-04-20 17:00:00 2026-04-20 18:00:00 324.27 324.92 🟢 TRAIL WIN        0.63
2026-04-21 17:00:00 2026-04-22 17:00:00 324.24 324.89 🟢 TRAIL WIN        0.27
2026-04-26 12:00:00 2026-04-27 09:00:00 325.84 324.23      🔴 LOSS       -1.00
2026-05-14 17:00:00 2026-05-15 17:00:00 325.82 324.00      🔴 LOSS       -1.00
2026-05-20 08:00:00 2026-05-20 09:00:00 324.39 323.40      🔴 LOSS       -1.00
2026-07-31 08:00:00 2026-07-31 09:00:00 274.23 274.27 🟢 TRAIL WIN        0.03
2026-08-06 15:00:00 2026-08-06 18:00:00 284.52 284.56 🟢 TRAIL WIN        0.01

📊 ИТОГИ БЭКТЕСТА (SMART MANAGEMENT): SBER
Настройки: Б/У при +2.0R | Трейлинг: True | Тейк: 1R
Всего сделок: 108
Закрыто по полному Тейку: 20
Закрыто по Трейлинг-стопу (в плюс): 31
Закрыто по Безубытку (в ноль): 0
Полные убытки: 

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. НАСТРОЙКИ БЭКТЕСТА
# ==========================================
TICKER = "SBER"
DAYS_BACK = 3 * 365
INTERVAL = 60
RR_RATIO = 2.0         # Классический риск-ревард 1:2

# ==========================================
# 2. ФИЛЬТРЫ ВХОДА (УЛУЧШЕНИЕ КАЧЕСТВА СИГНАЛА)
# ==========================================
# 1. Фильтр угла наклона (Slope)
USE_SLOPE_FILTER = True
MIN_SLOPE_PCT = 0.05   # EMA50 должна вырасти минимум на 0.05% за 5 баров

# 2. Фильтр перепроданности (RSI)
USE_RSI_FILTER = True
RSI_THRESHOLD = 45     # В фазе отката RSI должен был опуститься ниже 45

# 3. Фильтр объема (Volume)
USE_VOLUME_FILTER = True
VOL_MULTIPLIER = 1.2   # Объем зеленой пробойной свечи должен быть на 20% выше среднего

# ==========================================
# ФУНКЦИИ (Загрузка и Индикаторы)
# ==========================================
def fetch_history(ticker, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
    all_rows, start, columns = [], 0, None
    with tqdm(desc="Скачивание истории", leave=False) as pbar:
        while True:
            params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": interval, "start": start, "iss.meta": "off"}
            success = False
            for _ in range(3):
                try:
                    r = requests.get(url, params=params, timeout=15)
                    if r.status_code == 200:
                        success = True
                        break
                    time.sleep(2)
                except: time.sleep(2)
            if not success: break
            js = r.json().get("candles", {})
            rows = js.get("data", [])
            if columns is None: columns = js.get("columns", [])
            if not rows: break
            all_rows.extend(rows)
            pbar.update(1)
            if len(rows) < 500: break
            start += len(rows)
            time.sleep(0.1)

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date"})
    return df[["Date", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

def add_indicators(df):
    # Скользящие
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()

    # RSI (14)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = -delta.clip(upper=0).ewm(alpha=1/14, adjust=False).mean()
    df["RSI"] = 100 - (100 / (1 + gain / loss))

    # Средний объем за 20 баров
    df["Vol_SMA"] = df["Volume"].rolling(window=20).mean()

    # Наклон EMA50 (Изменение за 5 баров в %)
    df["EMA50_Slope"] = (df["EMA50"] - df["EMA50"].shift(5)) / df["EMA50"].shift(5) * 100

    return df

# ==========================================
# 3. ЛОГИКА СТРАТЕГИИ
# ==========================================
def run_backtest(df):
    trades = []
    in_trade, in_pullback = False, False
    pullback_low = float('inf')
    pullback_min_rsi = 100  # Для трекинга перепроданности в откате

    entry_price, sl, tp = 0, 0, 0
    entry_date = None

    # Статистика отсева (сколько сделок спасли фильтры)
    skipped_slope, skipped_rsi, skipped_vol = 0, 0, 0

    for i in range(200, len(df)):
        current = df.iloc[i]

        # --- ЛОГИКА ВЫХОДА ---
        if in_trade:
            if current["Low"] <= sl:
                trades.append({"Вход": entry_date, "Выход": current["Date"], "Вход Цена": entry_price, "Результат": "🔴 LOSS", "R": -1.0})
                in_trade = False
            elif current["High"] >= tp:
                trades.append({"Вход": entry_date, "Выход": current["Date"], "Вход Цена": entry_price, "Результат": "🏆 WIN", "R": RR_RATIO})
                in_trade = False
            continue

        # --- ЛОГИКА ВХОДА ---
        if current["EMA50"] <= current["EMA200"]:
            in_pullback = False
            continue

        if not in_pullback:
            # Цена заходит в зону отката
            if current["Close"] < current["Open"] and current["Close"] < current["EMA50"] and current["Close"] > current["EMA200"]:
                in_pullback = True
                pullback_low = current["Low"]
                pullback_min_rsi = current["RSI"]
        else:
            # Обновляем минимумы отката
            pullback_low = min(pullback_low, current["Low"])
            pullback_min_rsi = min(pullback_min_rsi, current["RSI"])

            if current["Close"] < current["EMA200"]:
                in_pullback = False
                continue

            # ТРИГГЕР: Зеленая свеча закрывается выше EMA50
            if current["Close"] > current["Open"] and current["Close"] > current["EMA50"]:

                # ПРОГОНЯЕМ ЧЕРЕЗ ФИЛЬТРЫ
                pass_slope = current["EMA50_Slope"] >= MIN_SLOPE_PCT if USE_SLOPE_FILTER else True
                pass_rsi = pullback_min_rsi <= RSI_THRESHOLD if USE_RSI_FILTER else True
                pass_vol = current["Volume"] >= (current["Vol_SMA"] * VOL_MULTIPLIER) if USE_VOLUME_FILTER else True

                # Учет того, какой фильтр убил сделку (для статистики)
                if not pass_slope: skipped_slope += 1
                elif not pass_rsi: skipped_rsi += 1
                elif not pass_vol: skipped_vol += 1

                # Если все фильтры пройдены - ВХОДИМ
                if pass_slope and pass_rsi and pass_vol:
                    in_trade = True
                    in_pullback = False
                    entry_price = current["Close"]
                    entry_date = current["Date"]

                    risk = entry_price - pullback_low
                    if risk <= 0: risk = entry_price * 0.005
                    sl = entry_price - risk
                    tp = entry_price + (risk * RR_RATIO)
                else:
                    # Если сигнал забракован, отменяем откат, ждем новый заход
                    in_pullback = False

    return pd.DataFrame(trades), skipped_slope, skipped_rsi, skipped_vol

# ==========================================
# 4. ЗАПУСК И ОТЧЕТ
# ==========================================
print(f"🔄 Анализ стратегии с ПРОДВИНУТЫМИ ФИЛЬТРАМИ...")
df = fetch_history(TICKER, DAYS_BACK, INTERVAL)

if not df.empty:
    df = add_indicators(df)
    trades_df, skip_sl, skip_rsi, skip_vol = run_backtest(df)

    print("\n" + "="*50)
    print(f"📊 ИТОГИ БЭКТЕСТА: {TICKER}")
    print(f"Фильтры: Тренд: {USE_SLOPE_FILTER} | RSI: {USE_RSI_FILTER} | Объемы: {USE_VOLUME_FILTER}")
    print("="*50)

    # Статистика фильтров
    total_ignored = skip_sl + skip_rsi + skip_vol
    print(f"🛡️ Отфильтровано плохих сигналов: {total_ignored}")
    if total_ignored > 0:
        print(f"   - Из-за слабого тренда (Флэт): {skip_sl}")
        print(f"   - Из-за слабого отката (RSI): {skip_rsi}")
        print(f"   - Из-за малого объема: {skip_vol}")
    print("-" * 50)

    if trades_df.empty:
        print("После применения фильтров сделок не осталось.")
    else:
        total = len(trades_df)
        wins = len(trades_df[trades_df["Результат"] == "🏆 WIN"])
        losses = len(trades_df[trades_df["Результат"] == "🔴 LOSS"])
        winrate = (wins / total) * 100
        total_pnl_r = trades_df["R"].sum()

        print(f"Всего сделок совершено: {total}")
        print(f"Прибыльных (WIN): {wins}")
        print(f"Убыточных (LOSS): {losses}")
        print(f"Винрейт: {winrate:.1f}%")
        print(f"Риск/Прибыль: 1:{RR_RATIO}")
        print("="*50)
        print(f"💰 ЧИСТЫЙ ПРОФИТ: {total_pnl_r:.2f} R")
        print("="*50)

🔄 Анализ стратегии с ПРОДВИНУТЫМИ ФИЛЬТРАМИ...



📊 ИТОГИ БЭКТЕСТА: SBER
Фильтры: Тренд: True | RSI: True | Объемы: True
🛡️ Отфильтровано плохих сигналов: 177
   - Из-за слабого тренда (Флэт): 166
   - Из-за слабого отката (RSI): 5
   - Из-за малого объема: 6
--------------------------------------------------
Всего сделок совершено: 4
Прибыльных (WIN): 1
Убыточных (LOSS): 3
Винрейт: 25.0%
Риск/Прибыль: 1:2.0
💰 ЧИСТЫЙ ПРОФИТ: -1.00 R


In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. ГЛОБАЛЬНЫЕ НАСТРОЙКИ СЕТКИ ПАРАМЕТРОВ
# ==========================================
TICKERS = ["IMOEX", "SBER", "LKOH", "GAZP", "ROSN", "YNDX", "TCSG"]
DAYS_BACK = 3 * 365
INTERVAL = 60

# Сетка для перебора
RR_RATIOS = [1.5, 2.0, 2.5, 3.0]
BREAKEVEN_TRIGGERS = [0, 1.0, 1.5]   # 0 = Б/У выключен
TRAILING_DISTANCES = [0, 1.0, 1.5]   # 0 = Трейлинг выключен

# ==========================================
# 2. ФИЛЬТРЫ ВХОДА (СМЯГЧЕННЫЕ)
# ==========================================
# 1. Фильтр угла наклона (Slope)
USE_SLOPE_FILTER = True
MIN_SLOPE_PCT = 0.01   # Смягчили: тренд должен хоть немного смотреть вверх, а не быть плоским

# 2. Фильтр перепроданности (RSI)
USE_RSI_FILTER = True
RSI_THRESHOLD = 55     # Смягчили: в сильном тренде RSI на откате редко падает ниже 45-50. 55 - отличный порог.

# 3. Фильтр объема (Volume)
USE_VOLUME_FILTER = False  # ОТКЛЮЧИЛИ: На часовиках MOEX объем слишком шумит и режет хорошие сделки
VOL_MULTIPLIER = 1.0

# ==========================================
# 2. ФУНКЦИИ ЗАГРУЗКИ ДАННЫХ И ИНДИКАТОРОВ
# ==========================================
def fetch_history(ticker, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)

    if ticker == "IMOEX":
        url = f"https://iss.moex.com/iss/engines/stock/markets/index/boards/SNDX/securities/{ticker}/candles.json"
    else:
        url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"

    all_rows, start, columns = [], 0, None

    while True:
        params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": interval, "start": start, "iss.meta": "off"}
        success = False
        for _ in range(3):
            try:
                r = requests.get(url, params=params, timeout=15)
                if r.status_code == 200:
                    success = True
                    break
                time.sleep(2)
            except: time.sleep(2)
        if not success: break

        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None: columns = js.get("columns", [])
        if not rows: break

        all_rows.extend(rows)
        if len(rows) < 500: break
        start += len(rows)
        time.sleep(0.1)

    if not all_rows: return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date"})
    return df[["Date", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

def add_indicators(df):
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = -delta.clip(upper=0).ewm(alpha=1/14, adjust=False).mean()
    df["RSI"] = 100 - (100 / (1 + gain / loss))
    df["Vol_SMA"] = df["Volume"].rolling(window=20).mean()
    df["EMA50_Slope"] = (df["EMA50"] - df["EMA50"].shift(5)) / df["EMA50"].shift(5) * 100
    return df

# ==========================================
# 3. БЫСТРЫЙ ДВИЖОК БЭКТЕСТА
# ==========================================
def run_fast_backtest(data_dicts, rr_ratio, be_trigger, trail_dist):
    in_trade, in_pullback = False, False
    pullback_low = float('inf')
    pullback_min_rsi = 100
    entry_price, sl, tp, initial_risk = 0, 0, 0, 0
    highest_price = 0

    wins, losses, breakevens = 0, 0, 0
    total_pnl_r = 0.0

    for current in data_dicts:
        if in_trade:
            highest_price = max(highest_price, current["High"])
            current_profit_R = (highest_price - entry_price) / initial_risk

            # Логика Б/У
            if be_trigger > 0 and current_profit_R >= be_trigger:
                if sl < entry_price: sl = entry_price

            # Логика Трейлинга
            if trail_dist > 0 and current_profit_R > trail_dist:
                new_sl = highest_price - (initial_risk * trail_dist)
                if new_sl > sl: sl = new_sl

            # Выход по стопу/БУ/трейлингу
            if current["Low"] <= sl:
                if sl == entry_price:
                    breakevens += 1
                elif sl > entry_price:
                    wins += 1
                    total_pnl_r += (sl - entry_price) / initial_risk
                else:
                    losses += 1
                    total_pnl_r -= 1.0
                in_trade = False

            # Выход по жесткому Тейку
            elif current["High"] >= tp:
                wins += 1
                total_pnl_r += rr_ratio
                in_trade = False

            continue

        # Логика Входа
        if current["EMA50"] <= current["EMA200"]:
            in_pullback = False
            continue

        if not in_pullback:
            if current["Close"] < current["Open"] and current["Close"] < current["EMA50"] and current["Close"] > current["EMA200"]:
                in_pullback = True
                pullback_low = current["Low"]
                pullback_min_rsi = current["RSI"]
        else:
            pullback_low = min(pullback_low, current["Low"])
            pullback_min_rsi = min(pullback_min_rsi, current["RSI"])

            if current["Close"] < current["EMA200"]:
                in_pullback = False
                continue

            if current["Close"] > current["Open"] and current["Close"] > current["EMA50"]:

                pass_slope = current["EMA50_Slope"] >= MIN_SLOPE_PCT if USE_SLOPE_FILTER else True
                pass_rsi = pullback_min_rsi <= RSI_THRESHOLD if USE_RSI_FILTER else True
                # Для индекса объем не всегда работает корректно, поэтому для IMOEX фильтр объема отключаем
                pass_vol = current["Volume"] >= (current["Vol_SMA"] * VOL_MULTIPLIER) if (USE_VOLUME_FILTER and current["Volume"] > 0) else True

                if pass_slope and pass_rsi and pass_vol:
                    in_trade = True
                    in_pullback = False
                    entry_price = current["Close"]

                    initial_risk = entry_price - pullback_low
                    if initial_risk <= 0: initial_risk = entry_price * 0.005

                    sl = entry_price - initial_risk
                    tp = entry_price + (initial_risk * rr_ratio)
                    highest_price = entry_price
                else:
                    in_pullback = False

    total_trades = wins + losses + breakevens
    winrate = (wins / total_trades * 100) if total_trades > 0 else 0
    return total_trades, winrate, total_pnl_r

# ==========================================
# 4. ГЛАВНЫЙ ЦИКЛ ПЕРЕБОРА (GRID SEARCH)
# ==========================================
print(f"🚀 ЗАПУСК МАССОВОГО БЭКТЕСТА (Grid Search)")
print(f"Тикеров: {len(TICKERS)} | Комбинаций на тикер: {len(RR_RATIOS) * len(BREAKEVEN_TRIGGERS) * len(TRAILING_DISTANCES)}")
print("="*70)

all_results = []

for ticker in tqdm(TICKERS, desc="Обработка тикеров"):
    # 1. Скачиваем данные ОДИН раз для тикера
    df = fetch_history(ticker, DAYS_BACK, INTERVAL)
    if df.empty or len(df) < 200:
        continue

    df = add_indicators(df)

    # Конвертируем в словари для бешеной скорости перебора
    data_dicts = df.iloc[200:].to_dict('records')

    # 2. Перебираем все параметры
    for rr in RR_RATIOS:
        for be in BREAKEVEN_TRIGGERS:
            for trail in TRAILING_DISTANCES:
                # Бессмысленно тестировать Б/У и Трейлинг, если их пороги больше Тейк-Профита
                if (be > 0 and be >= rr) or (trail > 0 and trail >= rr):
                    continue

                total, wr, pnl = run_fast_backtest(data_dicts, rr, be, trail)

                all_results.append({
                    "Тикер": ticker,
                    "Take (R)": rr,
                    "B/E (R)": be if be > 0 else "OFF",
                    "Trail (R)": trail if trail > 0 else "OFF",
                    "Сделок": total,
                    "Винрейт %": round(wr, 1),
                    "Профит (R)": round(pnl, 2)
                })

# ==========================================
# 5. ВЫВОД ТОП-РЕЗУЛЬТАТОВ
# ==========================================
if not all_results:
    print("❌ Нет результатов.")
else:
    results_df = pd.DataFrame(all_results)

    # Убираем связки, где сделок слишком мало (статистический шум)
    results_df = results_df[results_df["Сделок"] >= 3]

    # Сортируем по максимальному Профиту
    top_results = results_df.sort_values(by="Профит (R)", ascending=False).head(30)

    print("\n\n🏆 ТОП-30 ЛУЧШИХ КОМБИНАЦИЙ СТРАТЕГИИ:")
    print("="*80)
    print(top_results.to_string(index=False))

    print("\n" + "="*80)
    print("💡 КАК ЧИТАТЬ ТАБЛИЦУ:")
    print("Take (R): Где стоит тейк-профит (2.0 = прибыль в 2 раза больше стопа).")
    print("B/E (R): Когда стоп переносится в безубыток (OFF = отключено).")
    print("Trail (R): Дистанция трейлинг-стопа (OFF = отключено).")
    print("Профит (R): ЧИСТАЯ математическая прибыль стратегии за 3 года (с учетом фильтров входа!).")
    print("="*80)

🚀 ЗАПУСК МАССОВОГО БЭКТЕСТА (Grid Search)
Тикеров: 7 | Комбинаций на тикер: 36


Обработка тикеров: 100%|██████████| 7/7 [02:18<00:00, 19.79s/it]



🏆 ТОП-30 ЛУЧШИХ КОМБИНАЦИЙ СТРАТЕГИИ:
Тикер  Take (R) B/E (R) Trail (R)  Сделок  Винрейт %  Профит (R)
 LKOH       3.0     OFF       OFF      41       39.0       23.00
 LKOH       3.0     OFF       1.0      49       63.3       16.01
 LKOH       3.0     1.0       1.0      49       63.3       16.01
 LKOH       3.0     1.5       1.0      49       63.3       16.01
 LKOH       2.5     OFF       1.0      49       63.3       15.83
 LKOH       2.5     1.0       1.0      49       63.3       15.83
 LKOH       2.5     1.5       1.0      49       63.3       15.83
 LKOH       2.5     OFF       OFF      44       38.6       15.50
 TCSG       3.0     1.5       1.5       9       44.4       15.49
 TCSG       2.0     1.5       1.5       9       44.4       15.49
 TCSG       2.5     1.5       1.5       9       44.4       15.49
 TCSG       2.5     OFF       1.5       9       44.4       15.49
 TCSG       3.0     OFF       1.5       9       44.4       15.49
 TCSG       2.0     OFF       1.5       9       44

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. НАСТРОЙКИ СКРИНЕРА
# ==========================================
TOP_LIQUID_COUNT = 120  # Сколько самых ликвидных акций MOEX брать в тест
DAYS_BACK = 365         # Глубина истории — 1 год (чтобы быстро посчитать)
INTERVAL = 60           # Часовой таймфрейм

# Фильтры входа (остаются смягченными)
MIN_SLOPE_PCT = 0.01
RSI_THRESHOLD = 55

# 3 ЛУЧШИХ ПРЕСЕТА ИЗ ПРОШЛОГО ТЕСТА
PRESETS = [
    {"Name": "1. Макс Профит (TP 3.0)", "RR": 3.0, "BE": 0, "Trail": 0},
    {"Name": "2. Трейлинг (TP 3.0, Tr 1.0)", "RR": 3.0, "BE": 0, "Trail": 1.0},
    {"Name": "3. Умеренный (TP 2.0, BE 1.0, Tr 1.0)", "RR": 2.0, "BE": 1.0, "Trail": 1.0}
]

# ==========================================
# 2. ПОЛУЧЕНИЕ СПИСКА САМЫХ ЛИКВИДНЫХ АКЦИЙ
# ==========================================
def get_top_liquid_tickers(top_n):
    print("📊 Запрашиваем с MOEX список самых ликвидных акций...")
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,VALTODAY",
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        js = r.json()
        sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
        md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])

        df = sec.merge(md, on="SECID", how="left")
        df["VALTODAY"] = pd.to_numeric(df["VALTODAY"], errors="coerce").fillna(0)

        # Отсеиваем фонды ликвидности (LQDT, AKMM, SBMM и т.д.)
        ignore_list = ["LQDT", "AKMM", "SBMM", "BCSB", "TPAY"]
        df = df[~df["SECID"].isin(ignore_list)]

        df = df.sort_values("VALTODAY", ascending=False).reset_index(drop=True)
        return df.head(top_n)
    except Exception as e:
        print(f"❌ Ошибка загрузки списка акций: {e}")
        return pd.DataFrame()

# ==========================================
# 3. ЗАГУЗКА И ИНДИКАТОРЫ
# ==========================================
def fetch_history(ticker, days_back, interval):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
    all_rows, start, columns = [], 0, None

    while True:
        params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": interval, "start": start, "iss.meta": "off"}
        success = False
        for _ in range(3):
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code == 200:
                    success = True
                    break
                time.sleep(1)
            except: time.sleep(1)
        if not success: break

        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None: columns = js.get("columns", [])
        if not rows: break

        all_rows.extend(rows)
        if len(rows) < 500: break
        start += len(rows)
        time.sleep(0.05)

    if not all_rows: return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date"})
    return df[["Date", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").drop_duplicates(subset="Date").reset_index(drop=True)

def add_indicators(df):
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
    df["EMA200"] = df["Close"].ewm(span=200, adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = -delta.clip(upper=0).ewm(alpha=1/14, adjust=False).mean()
    df["RSI"] = 100 - (100 / (1 + gain / loss))
    df["EMA50_Slope"] = (df["EMA50"] - df["EMA50"].shift(5)) / df["EMA50"].shift(5) * 100
    return df

# ==========================================
# 4. ДВИЖОК БЭКТЕСТА
# ==========================================
def run_fast_backtest(data_dicts, rr_ratio, be_trigger, trail_dist):
    in_trade, in_pullback = False, False
    pullback_low = float('inf')
    pullback_min_rsi = 100
    entry_price, sl, tp, initial_risk = 0, 0, 0, 0
    highest_price = 0

    wins, losses, breakevens = 0, 0, 0
    total_pnl_r = 0.0

    for current in data_dicts:
        if in_trade:
            highest_price = max(highest_price, current["High"])
            current_profit_R = (highest_price - entry_price) / initial_risk

            if be_trigger > 0 and current_profit_R >= be_trigger:
                if sl < entry_price: sl = entry_price

            if trail_dist > 0 and current_profit_R > trail_dist:
                new_sl = highest_price - (initial_risk * trail_dist)
                if new_sl > sl: sl = new_sl

            if current["Low"] <= sl:
                if sl == entry_price: breakevens += 1
                elif sl > entry_price:
                    wins += 1
                    total_pnl_r += (sl - entry_price) / initial_risk
                else:
                    losses += 1
                    total_pnl_r -= 1.0
                in_trade = False
            elif current["High"] >= tp:
                wins += 1
                total_pnl_r += rr_ratio
                in_trade = False
            continue

        if current["EMA50"] <= current["EMA200"]:
            in_pullback = False
            continue

        if not in_pullback:
            if current["Close"] < current["Open"] and current["Close"] < current["EMA50"] and current["Close"] > current["EMA200"]:
                in_pullback = True
                pullback_low = current["Low"]
                pullback_min_rsi = current["RSI"]
        else:
            pullback_low = min(pullback_low, current["Low"])
            pullback_min_rsi = min(pullback_min_rsi, current["RSI"])
            if current["Close"] < current["EMA200"]:
                in_pullback = False
                continue

            if current["Close"] > current["Open"] and current["Close"] > current["EMA50"]:
                pass_slope = current["EMA50_Slope"] >= MIN_SLOPE_PCT
                pass_rsi = pullback_min_rsi <= RSI_THRESHOLD

                if pass_slope and pass_rsi:
                    in_trade = True
                    in_pullback = False
                    entry_price = current["Close"]
                    initial_risk = entry_price - pullback_low
                    if initial_risk <= 0: initial_risk = entry_price * 0.005
                    sl = entry_price - initial_risk
                    tp = entry_price + (initial_risk * rr_ratio)
                    highest_price = entry_price
                else:
                    in_pullback = False

    total_trades = wins + losses + breakevens
    winrate = (wins / total_trades * 100) if total_trades > 0 else 0
    return total_trades, winrate, total_pnl_r

# ==========================================
# 5. ГЛАВНЫЙ ИСПОЛНИТЕЛЬНЫЙ БЛОК
# ==========================================
stocks_df = get_top_liquid_tickers(TOP_LIQUID_COUNT)

if stocks_df.empty:
    print("❌ Не удалось получить тикеры.")
else:
    print(f"✅ Успешно загружен ТОП-{len(stocks_df)} акций. Начинаем массовое сканирование за 1 год...")
    print("="*80)

    screener_results = []

    for idx, row in tqdm(stocks_df.iterrows(), total=len(stocks_df), desc="Сканирование рынка"):
        ticker = row["SECID"]
        name = row["SHORTNAME"]

        df = fetch_history(ticker, DAYS_BACK, INTERVAL)
        if df.empty or len(df) < 200:
            continue

        df = add_indicators(df)
        data_dicts = df.iloc[200:].to_dict('records')

        # Прогоняем акцию по 3 лучшим пресетам
        for p in PRESETS:
            total, wr, pnl = run_fast_backtest(data_dicts, p["RR"], p["BE"], p["Trail"])

            # Фильтруем результаты (берём только там, где было хотя бы 5 сделок за год)
            if total >= 5 and pnl > 0: # Нам интересны только прибыльные!
                screener_results.append({
                    "Тикер": ticker,
                    "Компания": name,
                    "Прессет": p["Name"],
                    "Сделок за год": total,
                    "Винрейт %": round(wr, 1),
                    "Профит (R)": round(pnl, 2)
                })

    # ==========================================
    # 6. ИТОГОВЫЙ РЕЙТИНГ
    # ==========================================
    print("\n" + "="*80)
    if not screener_results:
        print("❌ Ни одна акция не показала прибыль с текущими настройками.")
    else:
        results_df = pd.DataFrame(screener_results)

        # Сортируем по максимальному чистому профиту
        top_stocks = results_df.sort_values(by="Профит (R)", ascending=False).reset_index(drop=True)

        print("🏆 ТОП АКЦИЙ MOEX, ИДЕАЛЬНО ПОДХОДЯЩИХ ДЛЯ СТРАТЕГИИ (За 1 год):")
        print("="*80)
        print(top_stocks.head(40).to_string(index=False))

        print("\n" + "="*80)
        print("💡 КАК ИСПОЛЬЗОВАТЬ ЭТОТ СПИСОК:")
        print("1. Акции вверху таблицы — ваши главные кандидаты для торговли на ближайшие месяцы.")
        print("2. Смотрите на колонку 'Прессет' — она говорит, какой именно стиль выходить работает лучше всего для конкретной акции.")
        print("3. Акций, которых нет в списке (или они не вышли в плюс), избегайте — они склонны к пилам и флэту.")
        print("="*80)

📊 Запрашиваем с MOEX список самых ликвидных акций...
✅ Успешно загружен ТОП-120 акций. Начинаем массовое сканирование за 1 год...


Сканирование рынка: 100%|██████████| 120/120 [13:17<00:00,  6.65s/it]


🏆 ТОП АКЦИЙ MOEX, ИДЕАЛЬНО ПОДХОДЯЩИХ ДЛЯ СТРАТЕГИИ (За 1 год):
Тикер   Компания                               Прессет  Сделок за год  Винрейт %  Профит (R)
 YDEX     ЯНДЕКС               1. Макс Профит (TP 3.0)             19       42.1       13.00
 UGLD        ЮГК               1. Макс Профит (TP 3.0)             19       42.1       13.00
 MTLR   Мечел ао               1. Макс Профит (TP 3.0)             19       42.1       13.00
LSNGP  РСетиЛЭ-п               1. Макс Профит (TP 3.0)             32       34.4       12.00
 HYDR   РусГидро 3. Умеренный (TP 2.0, BE 1.0, Tr 1.0)             23       52.2       11.91
SNGSP Сургнфгз-п          2. Трейлинг (TP 3.0, Tr 1.0)             24       50.0       11.72
SNGSP Сургнфгз-п 3. Умеренный (TP 2.0, BE 1.0, Tr 1.0)             24       50.0       11.72
 HYDR   РусГидро          2. Трейлинг (TP 3.0, Tr 1.0)             23       52.2       11.31
 VKCO МКПАО "ВК"               1. Макс Профит (TP 3.0)             13       46.2       11.00
 TATN

 ----

Откат после роста

In [4]:
# ==============================================================================
# GRID SEARCH: Поиск прибыльного управления для Шорта после +5% Пампа (2 ГОДА)
# ==============================================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. СЕТКА ПАРАМЕТРОВ СТОПОВ И ТЕЙКОВ
# ==========================================
DAYS_BACK = 2 * 365
PUMP_THRESHOLD_PCT = 5.0

# Варианты Стоп-Лосса
SL_MODES = [
    {"Name": "1. High T (Без запаса)", "type": "high_t", "add_pct": 0.0},
    {"Name": "2. High T + 1.5% (Запас)", "type": "high_t", "add_pct": 1.5},
    {"Name": "3. Фиксированный SL 2.5%", "type": "fixed", "add_pct": 2.5},
    {"Name": "4. Широкий SL 4.0%", "type": "fixed", "add_pct": 4.0},
]

# Варианты Тейк-Профита/Выхода
EXIT_MODES = [
    {"Name": "A. Закрытие дня T+1 (Без TP)", "type": "close_day", "val": 0.0},
    {"Name": "B. Быстрый Тейк 1:0.5 R", "type": "rr", "val": 0.5},
    {"Name": "C. Тейк 1:1.0 R", "type": "rr", "val": 1.0},
    {"Name": "D. Тейк 1:1.5 R", "type": "rr", "val": 1.5},
    {"Name": "E. Фикс Тейк +1.5% цены", "type": "fixed_tp", "val": 1.5},
]

# ==========================================
# 2. ЗАГРУЗКА ИСТОРИИ
# ==========================================
def get_all_tqbr_tickers():
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {"iss.meta": "off", "iss.only": "securities", "securities.columns": "SECID,SHORTNAME,STATUS"}
    for _ in range(3):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                js = r.json()
                df = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
                df = df[df["STATUS"] == "A"].reset_index(drop=True)
                ignore_list = ["LQDT", "AKMM", "SBMM", "BCSB", "TPAY"]
                return df[~df["SECID"].isin(ignore_list)]
        except: time.sleep(1)
    return pd.DataFrame()

def fetch_daily_candles_2y(ticker, days_back=730):
    till = datetime.now()
    since = till - timedelta(days=days_back)
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"
    all_rows, start, columns = [], 0, None
    while True:
        params = {"from": since.strftime("%Y-%m-%d"), "till": till.strftime("%Y-%m-%d"), "interval": 24, "start": start, "iss.meta": "off"}
        success = False
        for _ in range(3):
            try:
                r = requests.get(url, params=params, timeout=15)
                if r.status_code == 200:
                    success = True
                    break
                time.sleep(1)
            except: time.sleep(1)
        if not success: break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None: columns = js.get("columns", [])
        if not rows: break
        all_rows.extend(rows)
        if len(rows) < 500: break
        start += len(rows)
        time.sleep(0.05)
    if not all_rows: return pd.DataFrame()
    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"open": "Open", "close": "Close", "high": "High", "low": "Low", "volume": "Volume", "begin": "Date"})
    return df[["Date", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date").reset_index(drop=True)

# ==========================================
# 3. СБОР ВСЕХ СОБЫТИЙ ПАМПА (+5%)
# ==========================================
print("📊 Загружаем данные по всем акциям MOEX за 2 года...")
stocks_df = get_all_tqbr_tickers()
pump_events = []

for idx, row in tqdm(stocks_df.iterrows(), total=len(stocks_df), desc="Сбор памп-дней"):
    ticker = row["SECID"]
    df = fetch_daily_candles_2y(ticker, DAYS_BACK)
    if df.empty or len(df) < 100: continue

    df["Daily_Return_%"] = (df["Close"] - df["Close"].shift(1)) / df["Close"].shift(1) * 100

    for i in range(1, len(df) - 1):
        day_t = df.iloc[i]
        day_next = df.iloc[i+1]

        if day_t["Daily_Return_%"] >= PUMP_THRESHOLD_PCT:
            pump_events.append({
                "Ticker": ticker, "Date": day_t["Date"],
                "Pump_High": day_t["High"], "Pump_Close": day_t["Close"],
                "Open_N": day_next["Open"], "High_N": day_next["High"],
                "Low_N": day_next["Low"], "Close_N": day_next["Close"]
            })

print(f"\n✅ Собрано {len(pump_events)} памп-событий. Запускаем Grid Search параметров...")

# ==========================================
# 4. БЫСТРЫЙ GRID SEARCH
# ==========================================
grid_results = []

for sl_m in SL_MODES:
    for ex_m in EXIT_MODES:

        wins = 0
        losses = 0
        total_r = 0.0
        pnl_pct_sum = 0.0

        for ev in pump_events:
            entry = ev["Open_N"]

            # 1. Расчет Стоп-Лосса
            if sl_m["type"] == "high_t":
                sl_price = ev["Pump_High"] * (1 + sl_m["add_pct"] / 100)
            else: # fixed
                sl_price = entry * (1 + sl_m["add_pct"] / 100)

            risk = sl_price - entry

            # Если утренний гэп выбил стоп сразу на открытии
            if risk <= 0:
                losses += 1
                total_r -= 1.0
                pnl_pct_sum -= (ev["High_N"] - entry) / entry * 100
                continue

            # 2. Расчет Тейк-Профита
            if ex_m["type"] == "close_day":
                tp_price = 0 # Без Тейка, сидим до конца дня
            elif ex_m["type"] == "rr":
                tp_price = entry - (risk * ex_m["val"])
            elif ex_m["type"] == "fixed_tp":
                tp_price = entry * (1 - ex_m["val"] / 100)

            # 3. Симуляция сделки внутри дня T+1
            hit_sl = ev["High_N"] >= sl_price
            hit_tp = (tp_price > 0) and (ev["Low_N"] <= tp_price)

            if ex_m["type"] == "close_day":
                if hit_sl:
                    losses += 1
                    total_r -= 1.0
                    pnl_pct_sum -= (sl_price - entry) / entry * 100
                else:
                    # Выходим в конце дня
                    trade_pnl_pct = (entry - ev["Close_N"]) / entry * 100
                    trade_r = trade_pnl_pct / ((sl_price - entry) / entry * 100)
                    if trade_r > 0: wins

📊 Загружаем данные по всем акциям MOEX за 2 года...


Сбор памп-дней: 100%|██████████| 496/496 [09:16<00:00,  1.12s/it]


✅ Собрано 3729 памп-событий. Запускаем Grid Search параметров...


In [5]:
# ==============================================================================
# ИСПРАВЛЕННЫЙ GRID SEARCH (Запусти в новой ячейке, ничего качать не надо!)
# ==============================================================================

grid_results = []

for sl_m in SL_MODES:
    for ex_m in EXIT_MODES:

        wins = 0
        losses = 0
        total_r = 0.0
        pnl_pct_sum = 0.0

        for ev in pump_events:
            entry = ev["Open_N"]

            # 1. Расчет Стоп-Лосса
            if sl_m["type"] == "high_t":
                sl_price = ev["Pump_High"] * (1 + sl_m["add_pct"] / 100)
            else: # fixed
                sl_price = entry * (1 + sl_m["add_pct"] / 100)

            # Защита от деления на ноль!
            risk = sl_price - entry
            if risk <= 0:
                risk = entry * 0.005 # Если утренний гэп равен стопу, минимальный риск 0.5%
                sl_price = entry + risk

            risk_pct = (risk / entry) * 100

            # 2. Расчет Тейк-Профита
            if ex_m["type"] == "close_day":
                tp_price = 0
            elif ex_m["type"] == "rr":
                tp_price = entry - (risk * ex_m["val"])
            elif ex_m["type"] == "fixed_tp":
                tp_price = entry * (1 - ex_m["val"] / 100)

            # 3. Симуляция сделки
            hit_sl = ev["High_N"] >= sl_price
            hit_tp = (tp_price > 0) and (ev["Low_N"] <= tp_price)

            if ex_m["type"] == "close_day":
                if hit_sl:
                    losses += 1
                    total_r -= 1.0
                    pnl_pct_sum -= risk_pct
                else:
                    trade_pnl_pct = (entry - ev["Close_N"]) / entry * 100
                    trade_r = trade_pnl_pct / risk_pct
                    if trade_r > 0: wins += 1
                    else: losses += 1
                    total_r += trade_r
                    pnl_pct_sum += trade_pnl_pct
            else:
                if hit_sl and hit_tp:
                    losses += 1
                    total_r -= 1.0
                elif hit_sl:
                    losses += 1
                    total_r -= 1.0
                elif hit_tp:
                    wins += 1
                    total_r += ex_m["val"] if ex_m["type"] == "rr" else ((entry - tp_price) / risk)
                else:
                    trade_pnl_pct = (entry - ev["Close_N"]) / entry * 100
                    trade_r = trade_pnl_pct / risk_pct
                    if trade_r > 0: wins += 1
                    else: losses += 1
                    total_r += trade_r
                    pnl_pct_sum += trade_pnl_pct

        total_trades = wins + losses
        winrate = (wins / total_trades * 100) if total_trades > 0 else 0

        grid_results.append({
            "Стоп-Лосс": sl_m["Name"],
            "Вариант Выхода": ex_m["Name"],
            "Винрейт %": round(winrate, 1),
            "Профит (R)": round(total_r, 2),
            "Ср. Профит на сделку %": round(pnl_pct_sum / total_trades if total_trades > 0 else 0, 2)
        })

results_df = pd.DataFrame(grid_results).sort_values(by="Профит (R)", ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("🏆 РЕЗУЛЬТАТЫ СЕТКИ ПАРАМЕТРОВ ДЛЯ ШОРТА (От лучших к худшим):")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)


🏆 РЕЗУЛЬТАТЫ СЕТКИ ПАРАМЕТРОВ ДЛЯ ШОРТА (От лучших к худшим):
               Стоп-Лосс               Вариант Выхода  Винрейт %  Профит (R)  Ср. Профит на сделку %
  1. High T (Без запаса) A. Закрытие дня T+1 (Без TP)       32.9     1662.37                    0.23
2. High T + 1.5% (Запас) A. Закрытие дня T+1 (Без TP)       44.3     1022.00                    0.23
3. Фиксированный SL 2.5% A. Закрытие дня T+1 (Без TP)       42.4      159.17                    0.11
      4. Широкий SL 4.0% A. Закрытие дня T+1 (Без TP)       50.2      148.24                    0.16
      4. Широкий SL 4.0%              D. Тейк 1:1.5 R       50.6       91.42                    0.55
      4. Широкий SL 4.0%              C. Тейк 1:1.0 R       51.5       -1.54                    0.16
3. Фиксированный SL 2.5%              D. Тейк 1:1.5 R       43.3      -94.67                    0.20
2. High T + 1.5% (Запас)              D. Тейк 1:1.5 R       45.2     -195.11                    0.43
      4. Широкий SL 4.0%    